# Aula 13 - Figuras Ilustrativas dos Algoritmos

Este notebook gera figuras que mostram **como cada algoritmo funciona internamente**, não apenas os resultados.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.utils import PlotlyJSONEncoder
from pathlib import Path
import json
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def apply_dracula(fig, title=None):
    fig.update_layout(
        title=title,
        paper_bgcolor=DRACULA['bg'],
        plot_bgcolor=DRACULA['bg'],
        font=dict(color=DRACULA['fg']),
        margin=dict(l=60, r=30, t=60, b=55),
        showlegend=True,
        legend=dict(orientation='h', y=-0.25),
    )
    fig.update_xaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    fig.update_yaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    return fig

def clean_none(obj):
    if isinstance(obj, dict):
        return {k: clean_none(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [clean_none(v) for v in obj]
    return obj

def slide_export_path():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks' and cwd.parent.name == 'ciencia-dados':
        return cwd.parent / 'images' / 'plotly' / 'aula13_algo_illustrations.js'
    return cwd / 'ciencia-dados' / 'images' / 'plotly' / 'aula13_algo_illustrations.js'

def write_plotly_js(figures, output_path):
    lines = []
    for name, fig in figures.items():
        payload = clean_none(fig.to_dict())
        traces = json.dumps(payload['data'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        layout = json.dumps(payload['layout'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        lines.append(f"const {name}_TRACES = {traces};\n")
        lines.append(f"const {name}_LAYOUT = {layout};\n")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines), encoding='utf-8')

## 1. Z-score: Distribuição Normal com Desvios-Padrão

Mostra visualmente como o Z-score mede a distância em desvios-padrão.

In [ ]:
mu, sigma = 500, 120
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
y = stats.norm.pdf(x, mu, sigma)

fig_zscore_dist = go.Figure()

# Curva normal
fig_zscore_dist.add_trace(go.Scatter(
    x=x, y=y, mode='lines', name='Distribuição normal',
    line=dict(color=DRACULA['cyan'], width=3),
    fill='tozeroy', fillcolor=f'rgba(139,233,253,0.1)',
))

# Linhas de sigma
for mult, color, label in [(-3, DRACULA['red'], '-3σ'), (-2, DRACULA['yellow'], '-2σ'), (-1, DRACULA['muted'], '-1σ'),
                           (0, DRACULA['fg'], 'μ'), (1, DRACULA['muted'], '+1σ'), (2, DRACULA['yellow'], '+2σ'), (3, DRACULA['red'], '+3σ')]:
    fig_zscore_dist.add_vline(x=mu + mult*sigma, line_color=color, line_dash='dash' if mult != 0 else 'solid', line_width=2,
                              annotation_text=label, annotation_font_color=color, annotation_font_size=12)

# Regiões de anomalia
x_left = np.linspace(mu - 4*sigma, mu - 3*sigma, 100)
y_left = stats.norm.pdf(x_left, mu, sigma)
fig_zscore_dist.add_trace(go.Scatter(
    x=x_left, y=y_left, mode='lines', name='Região anômala (|z|>3)',
    line=dict(color=DRACULA['red'], width=2),
    fill='tozeroy', fillcolor=f'rgba(255,85,85,0.3)',
))

x_right = np.linspace(mu + 3*sigma, mu + 4*sigma, 100)
y_right = stats.norm.pdf(x_right, mu, sigma)
fig_zscore_dist.add_trace(go.Scatter(
    x=x_right, y=y_right, mode='lines', showlegend=False,
    line=dict(color=DRACULA['red'], width=2),
    fill='tozeroy', fillcolor=f'rgba(255,85,85,0.3)',
))

fig_zscore_dist.update_xaxes(title="Valor (gasto mensal R$)")
fig_zscore_dist.update_yaxes(title="Densidade")
fig_zscore_dist = apply_dracula(fig_zscore_dist, "Z-score: regiões de anomalia (|z| > 3) em vermelho")
fig_zscore_dist.show()

## 2. IQR: Quartis Visualizados

Mostra Q1, Q3, IQR e os limites de detecção.

In [ ]:
np.random.seed(42)
data = np.random.normal(500, 120, 300)
data = np.append(data, [1200, 1400, 100, 1600, 80])

q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr
median = np.median(data)

fig_iqr_viz = go.Figure()

# Strip plot dos dados
fig_iqr_viz.add_trace(go.Scatter(
    x=data, y=np.zeros_like(data), mode='markers', name='Dados',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.5),
))

# Linhas dos quartis
for val, color, label in [(q1, DRACULA['purple'], f'Q1={q1:.0f}'),
                          (median, DRACULA['fg'], f'Mediana={median:.0f}'),
                          (q3, DRACULA['pink'], f'Q3={q3:.0f}')]:
    fig_iqr_viz.add_vline(x=val, line_color=color, line_width=3,
                          annotation_text=label, annotation_font_color=color)

# IQR box
fig_iqr_viz.add_shape(type='rect', x0=q1, x1=q3, y0=-0.3, y1=0.3,
                      fillcolor=f'rgba(189,147,249,0.2)', line_color=DRACULA['purple'], line_width=2)
fig_iqr_viz.add_annotation(x=(q1+q3)/2, y=0.4, text=f'IQR = {iqr:.0f}',
                           showarrow=False, font=dict(color=DRACULA['purple'], size=14))

# Limites
for val, label in [(lim_inf, f'Lim inf={lim_inf:.0f}'), (lim_sup, f'Lim sup={lim_sup:.0f}')]:
    fig_iqr_viz.add_vline(x=val, line_color=DRACULA['red'], line_dash='dot', line_width=2,
                          annotation_text=label, annotation_font_color=DRACULA['red'])

# Outliers destacados
outliers = data[(data < lim_inf) | (data > lim_sup)]
fig_iqr_viz.add_trace(go.Scatter(
    x=outliers, y=np.zeros_like(outliers), mode='markers', name='Outlier IQR',
    marker=dict(color=DRACULA['red'], size=14, symbol='x', line=dict(width=3)),
))

fig_iqr_viz.update_yaxes(visible=False, range=[-0.5, 0.6])
fig_iqr_viz.update_xaxes(title="Gasto mensal (R$)")
fig_iqr_viz = apply_dracula(fig_iqr_viz, "IQR: quartis, intervalo interquartil e limites de detecção")
fig_iqr_viz.show()

## 3. Isolation Forest: Cortes Aleatórios

Mostra como cortes aleatórios isolam pontos. Pontos anômalos precisam de menos cortes.

In [ ]:
np.random.seed(42)
n = 80
normal_x = np.random.normal(5, 1.5, n)
normal_y = np.random.normal(5, 1.5, n)

anom_x = [1, 12, 13]
anom_y = [13, 1, 12]

fig_iso_cuts = go.Figure()

# Pontos normais
fig_iso_cuts.add_trace(go.Scatter(
    x=normal_x, y=normal_y, mode='markers', name='Pontos normais',
    marker=dict(color=DRACULA['cyan'], size=10),
))

# Pontos anômalos
fig_iso_cuts.add_trace(go.Scatter(
    x=anom_x, y=anom_y, mode='markers', name='Pontos anômalos',
    marker=dict(color=DRACULA['red'], size=16, symbol='star'),
))

# Cortes aleatórios (simulando uma iTree)
cuts = [
    {'type': 'v', 'pos': 9, 'color': DRACULA['yellow'], 'label': 'Corte 1'},
    {'type': 'h', 'pos': 9, 'color': DRACULA['pink'], 'label': 'Corte 2'},
    {'type': 'v', 'pos': 3, 'color': DRACULA['green'], 'label': 'Corte 3'},
    {'type': 'h', 'pos': 3, 'color': DRACULA['purple'], 'label': 'Corte 4'},
]

for i, cut in enumerate(cuts):
    if cut['type'] == 'v':
        fig_iso_cuts.add_vline(x=cut['pos'], line_color=cut['color'], line_width=2, line_dash='dash',
                               annotation_text=cut['label'], annotation_font_color=cut['color'])
    else:
        fig_iso_cuts.add_hline(y=cut['pos'], line_color=cut['color'], line_width=2, line_dash='dash',
                               annotation_text=cut['label'], annotation_font_color=cut['color'])

# Anotações explicativas
fig_iso_cuts.add_annotation(x=11, y=11, text='Anômalo isolado\nem 2 cortes!',
                            showarrow=True, arrowcolor=DRACULA['red'], font=dict(color=DRACULA['red'], size=13))
fig_iso_cuts.add_annotation(x=5, y=5, text='Normal ainda\nconectado',
                            showarrow=True, arrowcolor=DRACULA['fg'], font=dict(color=DRACULA['fg'], size=13))

fig_iso_cuts.update_xaxes(title="Variável X", range=[-1, 15])
fig_iso_cuts.update_yaxes(title="Variável Y", range=[-1, 15])
fig_iso_cuts = apply_dracula(fig_iso_cuts, "Isolation Forest: cortes aleatórios isolam anômalos mais rápido")
fig_iso_cuts.show()

## 4. Isolation Forest: Árvore de Isolamento

Mostra a estrutura da árvore com profundidade de isolamento.

In [ ]:
fig_iso_tree = go.Figure()

# Desenhar árvore manualmente com scatter + lines
# Root
nodes = {
    'root': (5, 10, 'Raiz\nX < 7?'),
    'l1': (2.5, 8, 'X < 3?'),
    'r1': (7.5, 8, 'Y < 6?'),
    'll': (1, 6, 'Folha\nh=2'),
    'lr': (4, 6, 'X < 5?'),
    'rl': (6.5, 6, 'Folha\nh=2'),
    'rr': (9, 6, 'Folha\nh=2'),
    'lrl': (3, 4, 'Folha\nh=3'),
    'lrr': (5, 4, 'Folha\nh=3'),
}

edges = [
    ('root', 'l1'), ('root', 'r1'),
    ('l1', 'll'), ('l1', 'lr'),
    ('r1', 'rl'), ('r1', 'rr'),
    ('lr', 'lrl'), ('lr', 'lrr'),
]

# Edges
for parent, child in edges:
    px, py, _ = nodes[parent]
    cx, cy, _ = nodes[child]
    fig_iso_tree.add_trace(go.Scatter(
        x=[px, cx], y=[py, cy], mode='lines',
        line=dict(color=DRACULA['muted'], width=2),
        showlegend=False, hoverinfo='skip',
    ))

# Nodes
nx, ny, labels = zip(*nodes.values())
fig_iso_tree.add_trace(go.Scatter(
    x=nx, y=ny, mode='markers+text', text=labels,
    marker=dict(color=DRACULA['cyan'], size=30, line=dict(color=DRACULA['fg'], width=2)),
    textposition='middle center', textfont=dict(color=DRACULA['fg'], size=10),
    name='Nós da árvore',
))

# Highlight leaf with h=2 (anomalous)
fig_iso_tree.add_trace(go.Scatter(
    x=[1], y=[6], mode='markers',
    marker=dict(color=DRACULA['red'], size=35, symbol='star', line=dict(width=2)),
    name='Folha rasa (anomalia)',
))

fig_iso_tree.update_xaxes(visible=False, range=[0, 11])
fig_iso_tree.update_yaxes(visible=False, range=[3, 11])
fig_iso_tree = apply_dracula(fig_iso_tree, "iTree: folhas rasas (h pequeno) = pontos anômalos")
fig_iso_tree.show()

## 5. LOF: k-Vizinhos Mais Próximos

Mostra como o LOF encontra os k vizinhos mais próximos de cada ponto.

In [ ]:
np.random.seed(42)
# Região densa
dense_x = np.random.normal(5, 0.8, 20)
dense_y = np.random.normal(5, 0.8, 20)

# Região esparsa
sparse_x = np.random.normal(10, 1.5, 8)
sparse_y = np.random.normal(10, 1.5, 8)

# Ponto de teste
test_x, test_y = 10, 10

fig_lof_neighbors = go.Figure()

# Região densa
fig_lof_neighbors.add_trace(go.Scatter(
    x=dense_x, y=dense_y, mode='markers', name='Região densa',
    marker=dict(color=DRACULA['cyan'], size=10),
))

# Região esparsa
fig_lof_neighbors.add_trace(go.Scatter(
    x=sparse_x, y=sparse_y, mode='markers', name='Região esparsa',
    marker=dict(color=DRACULA['yellow'], size=10),
))

# Ponto de teste
fig_lof_neighbors.add_trace(go.Scatter(
    x=[test_x], y=[test_y], mode='markers', name='Ponto p',
    marker=dict(color=DRACULA['red'], size=20, symbol='star'),
))

# k=5 vizinhos mais próximos (da região esparsa)
all_x = np.concatenate([dense_x, sparse_x, [test_x]])
all_y = np.concatenate([dense_y, sparse_y, [test_y]])
distances = np.sqrt((all_x - test_x)**2 + (all_y - test_y)**2)
k = 5
neighbor_indices = np.argsort(distances)[1:k+1]  # skip self

for idx in neighbor_indices:
    fig_lof_neighbors.add_trace(go.Scatter(
        x=[test_x, all_x[idx]], y=[test_y, all_y[idx]], mode='lines',
        line=dict(color=DRACULA['pink'], width=2, dash='dash'),
        showlegend=False,
    ))

# Círculo k-distância
k_dist = distances[neighbor_indices[-1]]
theta = np.linspace(0, 2*np.pi, 100)
fig_lof_neighbors.add_trace(go.Scatter(
    x=test_x + k_dist*np.cos(theta), y=test_y + k_dist*np.sin(theta),
    mode='lines', name=f'k-distância = {k_dist:.1f}',
    line=dict(color=DRACULA['purple'], width=2, dash='dot'),
))

fig_lof_neighbors.update_xaxes(title="Variável X", range=[0, 14])
fig_lof_neighbors.update_yaxes(title="Variável Y", range=[0, 14])
fig_lof_neighbors = apply_dracula(fig_lof_neighbors, f"LOF: k={k} vizinhos mais próximos do ponto p")
fig_lof_neighbors.show()

## 6. LOF: Comparação de Densidade Local

Mostra como o LOF compara a densidade do ponto com a dos vizinhos.

In [ ]:
fig_lof_density = go.Figure()

# Região densa
fig_lof_density.add_trace(go.Scatter(
    x=dense_x, y=dense_y, mode='markers', name='Região densa (LRD alto)',
    marker=dict(color=DRACULA['cyan'], size=10),
))

# Região esparsa
fig_lof_density.add_trace(go.Scatter(
    x=sparse_x, y=sparse_y, mode='markers', name='Região esparsa (LRD baixo)',
    marker=dict(color=DRACULA['yellow'], size=10),
))

# Ponto anômalo entre regiões
fig_lof_density.add_trace(go.Scatter(
    x=[7.5], y=[7.5], mode='markers', name='LOF alto (entre regiões)',
    marker=dict(color=DRACULA['red'], size=20, symbol='star'),
))

# Círculos de densidade
for cx, cy, r, color, label in [(5, 5, 2.5, DRACULA['cyan'], 'Densa'), (10, 10, 4, DRACULA['yellow'], 'Esparsa')]:
    theta = np.linspace(0, 2*np.pi, 100)
    fig_lof_density.add_trace(go.Scatter(
        x=cx + r*np.cos(theta), y=cy + r*np.sin(theta),
        mode='lines', name=f'{label}',
        line=dict(color=color, width=2, dash='dot'),
    ))

fig_lof_density.add_annotation(x=7.5, y=8.5, text='LOF >> 1\nmenos denso\nque vizinhos',
                               showarrow=True, arrowcolor=DRACULA['red'], font=dict(color=DRACULA['red'], size=13))

fig_lof_density.update_xaxes(title="Variável X", range=[0, 14])
fig_lof_density.update_yaxes(title="Variável Y", range=[0, 14])
fig_lof_density = apply_dracula(fig_lof_density, "LOF: compara densidade local com vizinhos")
fig_lof_density.show()

## 7. Score do Isolation Forest: Distribuição de Scores

Mostra a distribuição dos scores de anomalia para todos os pontos.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n_normal = 300
normais = pd.DataFrame({
    "gasto_mensal": np.random.normal(500, 120, n_normal),
    "numero_compras": np.random.normal(12, 3, n_normal)
})
anomalias_reais = pd.DataFrame({
    "gasto_mensal": [1200, 1400, 100, 1600, 80],
    "numero_compras": [2, 3, 40, 1, 45]
})
dados = pd.concat([normais, anomalias_reais], ignore_index=True)
dados["anomalia_real"] = [0] * n_normal + [1] * len(anomalias_reais)

X = dados[["gasto_mensal", "numero_compras"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
modelo.fit(X_scaled)

# decision_function retorna score: mais negativo = mais anômalo
scores = modelo.decision_function(X_scaled)
# Converter para score de anomalia (0 a 1)
anomaly_scores = -scores  # inverter: maior = mais anômalo
anomaly_scores = (anomaly_scores - anomaly_scores.min()) / (anomaly_scores.max() - anomaly_scores.min())

fig_iso_scores = go.Figure()

# Histograma dos scores
fig_iso_scores.add_trace(go.Histogram(
    x=anomaly_scores[dados["anomalia_real"] == 0], name='Pontos normais',
    marker=dict(color=DRACULA['cyan'], opacity=0.7), nbinsx=30,
))
fig_iso_scores.add_trace(go.Histogram(
    x=anomaly_scores[dados["anomalia_real"] == 1], name='Anomalias reais',
    marker=dict(color=DRACULA['red'], opacity=0.9), nbinsx=10,
))

# Linha de threshold
threshold = np.percentile(anomaly_scores, 95)
fig_iso_scores.add_vline(x=threshold, line_color=DRACULA['yellow'], line_width=2,
                         annotation_text='Threshold', annotation_font_color=DRACULA['yellow'])

fig_iso_scores.update_xaxes(title="Score de anomalia (0 = normal, 1 = anômalo)")
fig_iso_scores.update_yaxes(title="Frequência")
fig_iso_scores = apply_dracula(fig_iso_scores, "Isolation Forest: distribuição dos scores de anomalia")
fig_iso_scores.show()

## 8. LOF: Scores de todos os pontos

Mostra a distribuição dos scores LOF.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
pred_lof = modelo_lof.fit_predict(X_scaled)
lof_scores = -modelo_lof.negative_outlier_factor_  # maior = mais anômalo

fig_lof_scores = go.Figure()

fig_lof_scores.add_trace(go.Histogram(
    x=lof_scores[pred_lof == 1], name='Pontos normais (LOF ≈ 1)',
    marker=dict(color=DRACULA['cyan'], opacity=0.7), nbinsx=30,
))
fig_lof_scores.add_trace(go.Histogram(
    x=lof_scores[pred_lof == -1], name='Detectados como anomalia (LOF > 1)',
    marker=dict(color=DRACULA['red'], opacity=0.9), nbinsx=10,
))

fig_lof_scores.add_vline(x=1, line_color=DRACULA['yellow'], line_width=2,
                         annotation_text='LOF = 1 (normal)', annotation_font_color=DRACULA['yellow'])

fig_lof_scores.update_xaxes(title="Score LOF (≈1 = normal, >1 = anômalo)")
fig_lof_scores.update_yaxes(title="Frequência")
fig_lof_scores = apply_dracula(fig_lof_scores, "LOF: distribuição dos scores de densidade local")
fig_lof_scores.show()

## 9. Exportar Figuras

In [ ]:
figures = {
    'AULA13_ZSCORE_DIST': fig_zscore_dist,
    'AULA13_IQR_VIZ': fig_iqr_viz,
    'AULA13_ISO_CUTS': fig_iso_cuts,
    'AULA13_ISO_TREE': fig_iso_tree,
    'AULA13_LOF_NEIGHBORS': fig_lof_neighbors,
    'AULA13_LOF_DENSITY': fig_lof_density,
    'AULA13_ISO_SCORES': fig_iso_scores,
    'AULA13_LOF_SCORES': fig_lof_scores,
}

write_plotly_js(figures, slide_export_path())
print(f"Exported {len(figures)} figures to {slide_export_path()}")